# CS 431: Data-Intensive Distributed Computing (Fall 2026)
# Assignment 4: Machine Learning Pipelines

As a recap:

+ In Assignment 2, you started with raw data from the [Steam platform](https://store.steampowered.com/) and performed ETL and data cleaning.
+ In Assignment 3, you used Spark to answer questions using the data.

This assignment is about machine learning pipelines.

Imagine that you're still on the Steam data team.
The product team wants a simple classifier that identifies games with a high sampled-review positivity rate from their marketplace attributes.

You'll curate a training dataset, construct Spark ML pipelines, and compare logistic regression and a decision tree with a majority-class baseline.
The goal is a reproducible pipeline and a defensible evaluation, not extensive tuning.

You're expected to use a coding agent to help you with this assignment.
However, you are ultimately responsible for and accountable for _everything_, including design decisions, quality checks, bugs, edge cases, interpretation of the results, etc.
You are also responsible for _understanding_ what the coding agent is doing.

<font color="red"><b>Important Consideration</b></font>:
This assignment asks a number of short answer questions.
Answer completely but succinctly.
We don't expect an overly verbose explanation: simply enough to convey your understanding of the material.
LLMs often use a lot of words but convey little actual content.

Your coding agent will likely be able to one-shot the answer, but you should resist that temptation because it will likely impede your understanding if you simply ask the LLM to give you the answer.
As a suggestion, avoid simply copying/pasting the question into the prompt or directly pointing your agent at a specific notebook cell.
Try to come up with the answer yourself, perhaps using the agent to explain related concepts.
In other words, use the agent to help you _understand_ the material, not write the answer for you.
Such a strategy might be more conducive to learning.

**Start with a fresh database restore.**

This is to avoid possible confusion and leftover state from previous assignments.

+ Drop and recreate the database `steam_2025` under the user `cs431`.
+ Restore `steam_gold_a3.dump` in the database, from `https://cs.uwaterloo.ca/~jimmylin/cs431-2026f/steam_gold_a3.dump` with MD5 checksum `2821b99dccbcec9ff927e6299726299a`.
+ No database changes from your completed Assignment 3 notebook are required.

Everything below must be computed from tables in the `gold` schema in the above dump.
Do not modify the database.
When grading, we will provide a fresh restore; your notebook does not need restore commands.

Let's set up Spark. Run the following code. If you get an error, stop and debug.

In [ ]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable

# TODO: Set SPARK_HOME for your machine if PySpark is not already installed.
# os.environ["SPARK_HOME"] = "/path/to/spark-4.2.0-bin-hadoop3"
if os.getenv("SPARK_HOME"):
    import findspark
    findspark.init()

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.functions import vector_to_array

SEED = 12345
builder = (
    SparkSession.builder.appName("Assignment4").master("local[2]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
)

# Set POSTGRES_JDBC_JAR to a local driver file to avoid downloading the driver.
if os.getenv("POSTGRES_JDBC_JAR"):
    builder = builder.config("spark.jars", os.environ["POSTGRES_JDBC_JAR"])
else:
    builder = builder.config("spark.jars.packages", "org.postgresql:postgresql:42.7.7")

spark = builder.getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("spark_version=" + spark.version)

Use the following connection settings, as in Assignment 3.
Keep the defaults in your submission and supply credentials through environment variables.
Just run the cell.

In [ ]:
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = int(os.getenv("POSTGRES_PORT", "5432"))
POSTGRES_DB = os.getenv("POSTGRES_DB", "steam_2025")
POSTGRES_USER = os.getenv("POSTGRES_USER", "cs431")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "")
JDBC_URL = f"jdbc:postgresql://{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
JDBC_PROPERTIES = {
    "user": POSTGRES_USER, "password": POSTGRES_PASSWORD,
    "driver": "org.postgresql.Driver",
}

def read_table(table_name):
    if not table_name.replace("_", "").isalnum():
        raise ValueError("Unexpected table name")
    return spark.read.jdbc(JDBC_URL, f"gold.{table_name}", properties=JDBC_PROPERTIES)

def show_df(frame):
    print(frame.toPandas().to_string(index=False))

fact_games = read_table("fact_game_marketplace")
reviews = read_table("reviews").select("appid", "voted_up")

print("fact_game_n=" + str(fact_games.count()))
print("review_n=" + str(reviews.count()))

The counts should be `150274` and `865757`, respectively.
Don't continue until they are correct.

When doing this assignment:

+ Use Spark DataFrame transformations and Spark ML throughout.
+ Each required output name refers to a Spark DataFrame unless stated otherwise.
+ Use exactly the listed columns and ordering.
+ Preserve full precision except where rounding is explicitly requested.
+ Small aggregates, parameter tables, and final plots may use Python, NumPy, or Pandas; do not process training rows using these methods.
+ You may collect small final tables for printing, metrics, or inspecting fitted (trained) model parameters.
+ Do not collect the training rows to fit models in another library.

## 1. Building the Training Dataset

**Important:** The unit of observation is **one game**, not one review.
Include games with at least 20 sampled reviews and a non-NULL sampled positivity rate.
Set `highly_rated = 1` when that rate is at least `0.80`, and `0` otherwise.

We're going to be working on classification within a reviewed-game population using a single snapshot.
This is _not_ a forecast of a newly released game's future ratings.

### 1.1 Reconstruct the Target

As in Assignment 3, aggregate `reviews` by `appid` into `review_stats` with exactly:

+ `appid`
+ `derived_review_count`: number of review rows
+ `derived_positive_review_count`: number with `voted_up IS TRUE`
+ `derived_negative_review_count`: number with `voted_up IS FALSE`
+ `positive_review_rate`: positive count divided by total count, as a double; do not round

The dump has no `gold.dim_game`; the game name is already in `fact_games.name`.
Do not use the stored `review_positive_rate` to construct the label.

In [ ]:
## acell_411_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

review_stats

The above should create a `DataFrame[...]` with the specified columns.
Confirm.

You won't be surprised that the next thing we do is to sanity check what we've done.

Create `review_audit`, a one-row DataFrame with:

+ `reviewed_game_n`: rows in `review_stats`
+ `review_n`, `positive_review_n`, `negative_review_n`: sums of the three derived counts
+ `invalid_count_game_n`: games where total differs from positive plus negative

This checks the target's inputs before we fit anything.

In [ ]:
## acell_411_2 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(review_audit)

You should account for all `865757` reviews, and `invalid_count_game_n` should be zero.
If not, debug.

### 1.2 Choose Predictors and Preserve the Granularity

Use exactly the 18 candidate features below - they're provided, so just run the cell.

**Note**: Review counts, rates, playtime measures, identifiers, names, and review text are **audit-only**, not predictors.

In [ ]:
BOOLEAN_FEATURES = [
    "is_free_game", "supports_windows", "supports_mac", "supports_linux",
    "has_achievements", "missing_release_date", "future_release_date",
    "negative_price_flag", "invalid_achievement_count_flag",
]
NULLABLE_NUMERIC_FEATURES = ["price_usd", "release_year", "release_month", "achievement_count"]
COUNT_FEATURES = ["genre_count", "category_count", "developer_count", "publisher_count", "platform_count"]
CANDIDATE_FEATURES = [
    "price_usd", "is_free_game", "release_year", "release_month",
    "supports_windows", "supports_mac", "supports_linux", "achievement_count",
    "has_achievements", "genre_count", "category_count", "developer_count",
    "publisher_count", "platform_count", "missing_release_date", "future_release_date",
    "negative_price_flag", "invalid_achievement_count_flag",
]
FINAL_FEATURE_COLUMNS = [
    c + "_imputed" if c in NULLABLE_NUMERIC_FEATURES else c for c in CANDIDATE_FEATURES
]

Join `fact_games` with `review_stats`, apply the population rule, and create `labelled_games` with exactly, in this order:

+ `appid`, `game_name` (renamed from `name`)
+ `derived_review_count`, `positive_review_rate`, `highly_rated` (integer 0/1)
+ the 18 `CANDIDATE_FEATURES`, in their supplied order

Think about it (but no explicit question): What type of join (inner, outer, etc.) and why?

Keep original predictor types for now. Do not deduplicate after joining, sample rows, or drop games with missing predictors.
Set `labelled_n = labelled_games.count()` for subsequent audits.

In [ ]:
## acell_412_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

labelled_games

Surprise, we're going to sanity check the results!

Create `training_data_audit`, a one-row DataFrame with:

+ `fact_game_n`, `fact_distinct_appid_n`: fact row and distinct-ID counts
+ `labelled_game_n`, `labelled_distinct_appid_n`: corresponding labelled counts
+ `excluded_fact_game_n`: fact count minus labelled count
+ `incorrect_label_n`: labelled rows whose label disagrees with the unrounded `0.80` rule

Also create `target_distribution` with `highly_rated`, `game_n`, and `proportion` (game count divided by labelled count, rounded to four decimals). Order by label.

In [ ]:
## acell_412_2 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(training_data_audit)
print()
show_df(target_distribution)

There should be `10926` labelled games, with one row per `appid`, and no incorrect labels.

Create a DataFrame called `example_games` to pull some samples: the first six labelled rows ordered by `highly_rated`, then `appid`.
Select exactly `appid`, `game_name`, `derived_review_count`, `positive_review_rate`, `highly_rated`, and `price_usd`.
Keep the rate unrounded in this DataFrame.

In [ ]:
## acell_412_3 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(example_games)

**Q1.2(a): What exactly are we predicting?**

_Briefly_, respond to the following:

+ Identify the unit of observation, the target, and the eligible population.
+ Explain why it would make little sense to make a prediction using the positivity rate or the positive/negative counts.
+ Is total sampled review count alone sufficient to reconstruct the label?
+ Give one reason this evaluation cannot establish performance on newly released games.

// acell_412_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


## 2. Preprocessing to Reduce Test-Set Leakage

### 2.1 Inspect Missing Values

Create `feature_null_audit`, a DataFrame with one row per candidate feature and exactly:

+ `feature_index`: zero-based position in `CANDIDATE_FEATURES`
+ `feature_name`
+ `null_n`: NULL values among labelled games
+ `null_rate`: NULL count divided by labelled count, rounded to four decimals

Order by `feature_index`. Do not replace NULL values yet.

In [ ]:
## acell_421_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(feature_null_audit)

In this snapshot, the Boolean and count predictors have no NULL values.
Confirm this.
Do not silently replace an unexpected NULL with zero.

### 2.2 Split by Game Identifier

Create `model_data`, a DataFrame with exactly `appid`, `highly_rated`, all `CANDIDATE_FEATURES` in their supplied order, and `split_bucket`:

+ Cast every predictor to `double`, preserving NULL values. Boolean true/false become 1.0/0.0.
+ Set `split_bucket = pmod(xxhash64(appid, lit(12345)), 100)`.
+ Use the dump's `bigint` `appid` and an integer seed literal, exactly as above.

Create `train_data` from buckets below 80 (strictly `<`) and `test_data` from buckets at least 80 (`>=`).
Keep every `model_data` column in both splits.
Use `repartition(8, "appid").sortWithinPartitions("appid")`, then cache and materialize with `count()`.
Keep `labelled_n`, `train_n`, and `test_n` as Python integer row counts for later cells.

**Note:** Do not use `randomSplit`. We want you to do the above so that instance membership and the input layout are fixed for this exercise.

In [ ]:
## acell_422_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

print(f"train_n={train_n}, test_n={test_n}")

Yet another surprise, we're going to sanity check by auditing what we've done!

Create `split_audit`, a DataFrame with rows `test` and `train` and columns:
`split`, `game_n`, `positive_n` (sum of labels), and `positive_rate` (rounded to four decimals).
Order by `split`.

Also create `split_overlap_audit`, a one-row DataFrame with `overlap_appid_n`: distinct IDs shared by train and test.
The two populations should exhaust `labelled_games` and have no overlap.

In [ ]:
## acell_422_2 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(split_audit)
print()
show_df(split_overlap_audit)

Think about it (nothing to specifically answer): What value should `overlap_appid_n` be?
If it's not as you expected, stop and debug.

### 2.3 Learn the Preprocessing on Training Rows

We're next going to fit an imputer model using the `Imputer` in Spark ML.
The code is provided for you, just run the cell:

In [ ]:
imputer = Imputer(strategy="median", relativeError=0.0,
    inputCols=NULLABLE_NUMERIC_FEATURES,
    outputCols=[c + "_imputed" for c in NULLABLE_NUMERIC_FEATURES])
imputer_model = imputer.fit(train_data)
train_imputed = imputer_model.transform(train_data)
test_imputed = imputer_model.transform(test_data)

surrogates = imputer_model.surrogateDF.first()
imputation_values = spark.createDataFrame([
    (c, float(surrogates[c])) for c in NULLABLE_NUMERIC_FEATURES
], "feature_name string, training_median double").orderBy("feature_name")

show_df(imputation_values)

**Q2.3(a): What's the `Imputer` and what is the above code doing?**

Explain what an imputer is and what the code above is doing.
Why is this an important thing to do on the input features?

// acell_423_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


Let's confirm that the imputer did something.

Consider the field `release_year`: What is the field name of the imputed values?

Below, in the `train` split, compute the number of NULL values of `release_year` before imputation and after imputation.

In [ ]:
## acell_423_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

print(f'Count of NULL values of "release_year" before imputation: {before}')
print(f'Count of NULL values of "release_year" after imputation:  {after}')

So, it seems to have worked? Nothing to answer, but convince yourself that the results above make sense.

Now let's do a more formal audit.
We also need to check that imputation preserves observed values. 

Create `imputation_audit`, a DataFrame with the following columns:

+ `split`
+ `feature_name`
+ `missing_before_n`: count of NULL values before imputation
+ `missing_after_n`: count of NULL values after imputation
+ `changed_observed_n`: count where the original value is non-NULL and differs from the corresponding imputed value

Order by `split`, `feature_name`.

Things to think about (no need to answer explicitly):

+ `missing_after_n` and `changed_observed_n`: what values should these be?
+ How many rows should there be in this DataFrame?


In [ ]:
## acell_423_2 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(imputation_audit)

Okay, there is one final step for logistic regression.
We provide the code below, just run the cell:

In [ ]:
assembler = VectorAssembler(inputCols=FINAL_FEATURE_COLUMNS, outputCol="features_raw", handleInvalid="error")

train_vectors = assembler.transform(train_imputed)
test_vectors = assembler.transform(test_imputed)

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)
scaler_model = scaler.fit(train_vectors)

train_scaled = scaler_model.transform(train_vectors)
test_scaled = scaler_model.transform(test_vectors)

**Q2.3(b): What's a `Scaler` and what is the above code doing?**

Explain how the code assembles and scales the features. 
Why is feature scaling useful for logistic regression?

// acell_423_b (keep this id for tracking purposes)

**Answer:** Replace with your answer.


**Q2.3(c): Why do we not scale features for decision trees?**


// acell_423_c (keep this id for tracking purposes)

**Answer:** Replace with your answer.


**Q2.3(d): Why fit preprocessing on training data only?**


// acell_423_d (keep this id for tracking purposes)

**Answer:** Replace with your answer.


**Q2.3(e): Do you think there's something odd about treating `release_month` as a single numeric predictor in logistic regression?**


// acell_423_e (keep this id for tracking purposes)

**Answer:** Replace with your answer.


## 3. Training and Evaluating Classifiers

### 3.1 Construct Two Pipelines

We're finally done with data preparation and are actually going to train classifiers!

Use the supplied estimator settings below; do not tune them.

Note: Logistic regression uses ridge regularization because several predictors overlap (for example, platform count and platform flags).
External scaling is already specified, so internal logistic-regression `standardization` is disabled.

In [ ]:
logistic = LogisticRegression(
    labelCol="highly_rated", featuresCol="features", family="binomial",
    maxIter=100, tol=1e-8, regParam=0.1, elasticNetParam=0.0,
    fitIntercept=True, standardization=False, threshold=0.5,
)

tree = DecisionTreeClassifier(
    labelCol="highly_rated", featuresCol="features_raw", impurity="gini",
    maxDepth=5, maxBins=32, minInstancesPerNode=1, minInfoGain=0.0, seed=SEED,
)

Below, we're going to create two pipelines:

+ `logistic_pipeline` with stages `[imputer, assembler, scaler, logistic]`
+ `tree_pipeline` with `[imputer, assembler, tree]`

These are estimators, so the preprocessing will be learned inside each pipeline's training-only fit.
Fit each pipeline on **raw `train_data`**, creating `logistic_model` and `tree_model`.

Do not pass already transformed rows to these pipelines.

The code is provided for you, just run the cell.
However, make sure you understand what the code is doing.

Note: We fitted preprocessing separately to inspect its behavior; now we repeat it inside each pipeline so preprocessing and classification form one complete workflow.

In [ ]:
logistic_pipeline = Pipeline(stages=[imputer, assembler, scaler, logistic])
tree_pipeline = Pipeline(stages=[imputer, assembler, tree])

logistic_model = logistic_pipeline.fit(train_data)
tree_model = tree_pipeline.fit(train_data)

print("logistic_iterations=" + str(logistic_model.stages[-1].summary.totalIterations))
print("tree_depth=" + str(tree_model.stages[-1].depth))

**Q3.1(a): What does the above output mean?**

Explain what the above cell printed.

// acell_431_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


### 3.2 Establish a Baseline and Score the Test Set

Before making predictions, we need a baseline.

**Q3.2(a): Why is random guessing not the best baseline?**

There are two labels, 0/1, so the baseline should just be random guessing, 50/50, right?

// acell_432_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


So, what's a better baseline?
Look at the code below (just run the cell):

In [ ]:
majority_label = int(train_data.groupBy("highly_rated").count()
    .orderBy(F.desc("count"), F.asc("highly_rated")).first()["highly_rated"])
baseline_predictions = test_data.select("appid", "highly_rated").withColumn(
    "prediction", F.lit(float(majority_label)))

print("majority_label=" + str(majority_label))

**Q3.2(b): What's the code doing?**

Why is it a better baseline?
(Might be useful to define what "better" means.)

// acell_432_b (keep this id for tracking purposes)

**Answer:** Replace with your answer.


Now, let's make predictions using the trained (fitted) logistic regression and decision tree models.
Just run the cell below:

In [ ]:
logistic_predictions = logistic_model.transform(test_data).cache()
tree_predictions = tree_model.transform(test_data).cache()

Before evaluating, let us sanity check the predictions.

Create `prediction_audit`, a DataFrame with three rows (one row per model) and the following columns:

+ `model`
+ `prediction_n`
+ `distinct_appid_n`
+ `missing_test_appid_n`
+ `unexpected_appid_n`
+ `invalid_prediction_n`

The fields `prediction_n` and `distinct_appid_n` are row and distinct-ID counts; missing/unexpected compare IDs with `test_data`.
Invalid predictions are NULL or outside `{0.0, 1.0}`.

Use the three model names: `majority_baseline`,  `logistic_regression`, `decision_tree`.
Order by `model`.

In [ ]:
## acell_432_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(prediction_audit)

The first two counts should equal `test_n`.

What should the other values be?
No need to explicitly answer, but think it through.
Debug if the values are unexpected.

Let's pull some examples.

Create a DataFrame named `prediction_examples`, with six rows, populated with the six smallest test `appid` values.
The columns should be:

+ `appid`
+ `highly_rated`
+ `baseline_prediction`
+ `logistic_prediction`
+ `logistic_probability`
+ `tree_prediction`
+ `tree_probability`.

The column names should be self-explanatory.

Each probability is the model's probability of class 1 (`probability[1]`), rounded to four decimals.
Order by `appid`. Do not round probabilities before generating predictions.

In [ ]:
## acell_432_2 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(prediction_examples)

**Q3.2(c): What does it mean?**

Pick a row.
Explain what all the columns mean.

// acell_432_c (keep this id for tracking purposes)

**Answer:** Replace with your answer.


### 3.3 Analyze Classification Errors

Let's analyze the classification errors in terms of a standard 2-by-2 confusion matrix:

+ the actual true label, which can be 0 or 1
+ the prediction, which can also be 0 or 1

So, in total, we have four cases.

Create `confusion_counts`, a DataFrame with exactly 12 rows (the four cases above and three models).
The DataFrame should have exactly these columns:

+ `model`: `majority_baseline`, `logistic_regression`, or `decision_tree`
+ `label`: true label, integer 0 or 1
+ `prediction`: predicted label, integer 0 or 1
+ `count`: test games in that combination, including zero-count combinations

Order by `model`, `label`, `prediction`.

In [ ]:
## acell_433_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(confusion_counts)

Let's treat label 1 as positive.
This means that:

+ TP is `(1,1)`
+ TN `(0,0)`
+ FP `(0,1)`
+ FN `(1,0)`

From these counts, create `evaluation_metrics`, a DataFrame with three rows (one row per model) and the following columns:

+ `model`, `test_n`
+ `accuracy`: `(TP + TN) / test_n`
+ `precision_positive`: `TP / (TP + FP)`
+ `recall_positive`: `TP / (TP + FN)`
+ `f1_positive`: `2 * precision_positive * recall_positive / (precision_positive + recall_positive)`

Additional requirements:

+ Report 0 when a metric's denominator is zero.
+ Calculate with unrounded values, then round the four metrics to four decimals.
+ Order by `model`.
+ Compute the metrics yourself. Do not substitute Spark's weighted precision, recall, or F1.

In [ ]:
## acell_433_2 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(evaluation_metrics)

**Q3.3(a): Both the logistic regression and decision tree models outperform the majority baseline.**

Is the above claim empirically supported?
Why or why not?

Discuss from the perspective of the tradeoffs between different metrics.

// acell_433_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


**Q3.3(b): Logistic regression is a better model than decision trees.**

Is the above claim empirically supported?
Why or why not?

// acell_433_b (keep this id for tracking purposes)

**Answer:** Replace with your answer.


**Q3.3(c): False positives and false negatives have different consequences.**

For logistic regression, how many false positives (FPs) and false negatives (FNs) are there?
Explain in simple terms what each error type means.

Suppose the product team will recommend games predicted to be highly rated and wants to avoid recommending lower-rated games.
Which type of error is more important in this context?
Based on these results, which of the two trained models might be more suited?

// acell_433_c (keep this id for tracking purposes)

**Answer:** Replace with your answer.


### 3.4 Training Performance and Generalization

Apply the already fitted (trained) models to `train_data`; do not refit them.

Create `training_confusion_counts`, a DataFrame with the same columns and four-cell convention as `confusion_counts`, but only the two fitted (trained) models (eight rows). Order by `model`, `label`, `prediction`.

In [ ]:
## acell_434_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

# Use this:
training_predictions = {
    "logistic_regression": logistic_model.transform(train_data),
    "decision_tree": tree_model.transform(train_data),
}

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(training_confusion_counts)

Let's compare accuracy on training and test.

Create `generalization_metrics`, a DataFrame with two rows and the following columns:

+ `model`
+ `train_n`
+ `test_n`
+ `train_accuracy`
+ `test_accuracy`
+ `accuracy_gap`: `train_accuracy - test_accuracy`.

Compute accuracy from the corresponding confusion counts.
Compute the gap before rounding all three rates to four decimals. Order by `model`.

In [ ]:
## acell_434_2 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(generalization_metrics)

Plot the two models' training and test accuracy as grouped bars, using `generalization_metrics`.
Put models in that table's order on the x-axis, use a 0-to-1 y-axis, label axes, include a legend and descriptive title, and display the plot.

In [ ]:
## acell_434_3 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

**Q3.4(a): Wait, accuracy on the test set is higher than accuracy on the training set?**

How is this possible? Explain.


// acell_434_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


### 3.5 Inspect Features in the Fitted Model

Next, let's explore the features in the final fitted (trained) models.

Create `feature_inspection`, a DataFrame with one row per (final) feature, with exactly the following columns:

+ `feature_index`, `feature_name`
+ `logistic_coefficient`: coefficient on the externally scaled feature
+ `tree_importance`: decision-tree impurity-based importance at the same index

Keep values unrounded. Order by decreasing absolute coefficient, breaking ties by `feature_index` ascending.
The tree uses `features_raw`; both models have the same feature order.
The intercept is not a feature and must not appear in this table.

In [ ]:
## acell_435_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

show_df(feature_inspection)

Plot the first six rows of `feature_inspection` as horizontal bars of signed logistic coefficients.
Put the largest absolute coefficient at the top; preserve the table's tie rule.
Label features and the coefficient axis, include a vertical zero line and descriptive title, and display the plot.

In [ ]:
## acell_435_2 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

**Q3.5(a): Analyze the model based on the results above.**

What's the most important feature?

// acell_435_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


Let's try to similarly visualize the decision tree.

The decision tree can be visualized as a textual if-then-else tree.
Do that.
(You just need to find the right API call.)

In [ ]:
## acell_435_3 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########


######### END STUDENT CODE #########

**Q3.5(b): Analyze the model based on the results above.**

What's the most important feature according to the model's internal `tree_importance` measure?
What's the top-level node of the decision tree?
(Write out the decision rule in words.)

Are these the same?

// acell_435_b (keep this id for tracking purposes)

**Answer:** Replace with your answer.


**Q3.5(c): Are newer games just better?**

Both the logistic regression and decision tree models seem to agree that newer games are just better.
Is this correct?

// acell_435_c (keep this id for tracking purposes)

**Answer:** Replace with your answer.


All done with Spark! Stop the session.

In [ ]:
logistic_predictions.unpersist()
tree_predictions.unpersist()
train_data.unpersist()
test_data.unpersist()
labelled_games.unpersist()
spark.stop()

## 4. Reflection on AI Collaboration

Complete this log honestly. This is primarily for your own benefit; we'll read it, but it won't be graded.

+ What I asked the coding agent to do.
+ What worked and what needed correction.
+ One leakage, data-grain, or metric issue I checked myself.
+ One place where my statistical or lecture knowledge mattered.
+ One prompt I would write differently next time.

// acell_440_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


## 5. Assignment Submission

Before submitting:

+ Restart the kernel and run all cells from top to bottom against a fresh dump restore.
+ Keep every `acell_...` identifier and instructor-provided output statement unchanged.
+ Keep every required DataFrame with its specified name, columns, and full row set.
+ Keep missing-value handling, feature order, split expression, and model settings exactly as specified.
+ Remove temporary debugging output, but retain the required tables and written answers.
+ Submit the completed notebook according to the course instructions; no model files are required.